# **Hospital Readmission Prediction & Patient Risk Intelligence System**

### 1. Import Libraries and Loading Dataset

In [25]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/diabetic_data.csv")

### 2. Check Missing Values

In [26]:
df.isnull().sum()

,0
encounter_id,0
patient_nbr,0
race,0
gender,0
age,0
weight,0
admission_type_id,0
discharge_disposition_id,0
admission_source_id,0
time_in_hospital,0


### 3. Replace '?' With NaN

In [27]:
df.replace('?', np.nan, inplace=True)

### 4. Check Missing Values

In [28]:
missing_values = df.isnull().sum()

missing_values[missing_values > 0]

,0
race,2273
weight,98569
payer_code,40256
medical_specialty,49949
diag_1,21
diag_2,358
diag_3,1423
max_glu_serum,96420
A1Cresult,84748


### 5. Calculate Missing Percentage

In [29]:
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_percent.sort_values(ascending=False)
missing_percent[missing_percent > 0]

,0
race,2.233555
weight,96.858479
payer_code,39.557416
medical_specialty,49.082208
diag_1,0.020636
diag_2,0.351787
diag_3,1.398306
max_glu_serum,94.746772
A1Cresult,83.277322


### 6. Droping Columns which are not useful and which have more missing values

**1. Drop Weight**

97% missing means only 3% of patients have weight information.

The model cannot learn meaningful patterns from such sparse data.

In [30]:
df.drop(columns=['weight'], inplace=True)

**2. Drop Payer_Code**

This column represents insurance/payment information.

For readmission prediction, clinical features are more important.

In [31]:
df.drop(columns=['payer_code'], inplace=True)

**3. About max_glu_Serum and A1Cresult**

1.max_glu_serum

Represents the maximum serum glucose test result during hospitalization.

| Value | Meaning                        |
| ----- | ------------------------------ |
| None  | Test not performed             |
| Norm  | Normal glucose level           |
| >200  | Glucose greater than 200 mg/dL |
| >300  | Glucose greater than 300 mg/dL |

Patients with very high glucose levels often have:

Poor diabetes control
Severe complications
Higher chance of readmission

Therefore this feature can be highly predictive.

2.A1Cresult

Represents the HbA1c test result.

| Value | Meaning               |
| ----- | --------------------- |
| None  | Test not performed    |
| Norm  | Normal HbA1c          |
| >7    | HbA1c greater than 7% |
| >8    | HbA1c greater than 8% |

HbA1c indicates long-term blood sugar control over the previous 2–3 months.

Higher HbA1c generally indicates:

Poor diabetes management
Increased complications
Increased readmission probability

This makes it one of the medically meaningful features in the dataset.



### 7. Handle Medical_speciality

Medical specialty may contain useful information:

Examples:

Cardiology
Endocrinology
Neurology

Removing it may lose predictive power.

In [ ]:
df['medical_specialty'].fillna(
    'Unknown',
    inplace=True
)

### 8. Check Duplicate Rows

In [33]:
df.duplicated().sum()

np.int64(0)

Remove Duplicate

In [34]:
df.drop_duplicates(inplace=True)

### 9. Check Invalid Gender Values

In [35]:
df['gender'].value_counts()

,count
gender,
Female,54708
Male,47055
Unknown/Invalid,3


Remove Invalid Gender Records

In [36]:
df = df[df['gender'] != 'Unknown/Invalid']

### 10. Remove Identifier Columns

These columns uniquely identify patients.

Machine learning may memorize IDs instead of learning medical patterns.

This is called:

Data Leakage

and it can severely damage model generalization.

In [37]:
df.drop(
    columns=[
        'encounter_id',
        'patient_nbr'
    ],
    inplace=True
)

In [38]:
df.isnull().sum().sort_values(ascending=False)

,0
max_glu_serum,96417
A1Cresult,84745
race,2271
diag_3,1423
diag_2,358
diag_1,21
time_in_hospital,0
medical_specialty,0
num_lab_procedures,0
gender,0


### 11. Saving Cleaned Dataset

In [ ]:
import os
os.makedirs('/cleaned_data', exist_ok=True)
df.to_csv(
    "/cleaned_data/diabetic_cleaned.csv",
    index=False
)